In [1]:
# ============================================
# PHASE 4: EVALUATION ON UNSEEN DATA (FULL SET)
# ============================================

import torch
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# -----------------------------
# 1. LOAD TRAINED MODEL
# -----------------------------
MODEL_DIR = "/Users/rohan/Documents/Final year/cwe_project/scripts/codet5_juliet_multiclass"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR)

device = "mps" if torch.backends.mps.is_available() else "cpu"
model.to(device)

print("✅ Model & tokenizer loaded from:", MODEL_DIR)

# -----------------------------
# 2. LOAD THE FULL DATASET
# -----------------------------
CSV_PATH = "/Users/rohan/Documents/Final year/cwe_project/juliet_cwe_dataset.csv"
df = pd.read_csv(CSV_PATH)

# Only what we need
df = df[["code_clean", "cwe_index"]].dropna()
df["cwe_index"] = df["cwe_index"].astype(int)

print("📦 Loaded full dataset:", len(df))

# -----------------------------
# 3. DATASET CLASS
# -----------------------------
class JulietDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        code = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            code,
            truncation=True,
            padding="max_length",
            max_length=self.max_len
        )

        item = {k: torch.tensor(v) for k, v in encoding.items()}
        item["labels"] = torch.tensor(label)
        return item

# Create dataloader on whole dataset
full_dataset = JulietDataset(df["code_clean"], df["cwe_index"], tokenizer)
full_loader = DataLoader(full_dataset, batch_size=8 , shuffle=False)

# -----------------------------
# 4. EVALUATION FUNCTION
# -----------------------------
def evaluate(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=-1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return all_labels, all_preds

# -----------------------------
# 5. RUN EVALUATION
# -----------------------------
y_true, y_pred = evaluate(model, full_loader, device)

print("\n📊 Classification Report:")
print(classification_report(y_true, y_pred, digits=4))

print("\n📉 Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1219 13:20:25.222000 37127 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


✅ Model & tokenizer loaded from: /Users/rohan/Documents/Final year/cwe_project/scripts/codet5_juliet_multiclass
📦 Loaded full dataset: 179839


Evaluating:  29%|██▊       | 6432/22480 [20:18<50:40,  5.28it/s]   


KeyboardInterrupt: 

In [ ]:
# import pandas as pd
# import torch
# from sklearn.metrics import accuracy_score
# from torch.utils.data import DataLoader

# # Load full dataset
# df_full = pd.read_csv("/Users/rohan/Documents/Final year/cwe_project/juliet_cwe_dataset.csv")

# device = "mps" if torch.backends.mps.is_available() else "cpu"
# model.to(device)

# cwe_groups = df_full.groupby("cwe")

# folder_results = []

# for cwe, group in cwe_groups:
#     print(f"Evaluating CWE: {cwe}  ({len(group)} samples)")

#     texts = group["code_clean"].tolist()
#     labels = group["cwe_index"].tolist()

#     encodings = tokenizer(
#         texts,
#         truncation=True,
#         padding="max_length",
#         max_length=256,
#         return_tensors="pt"
#     )
    
#     encodings = {k: v.to(device) for k, v in encodings.items()}
#     labels_t = torch.tensor(labels).to(device)

#     with torch.no_grad():
#         logits = model(**encodings).logits
#         preds = torch.argmax(logits, dim=-1).cpu().numpy()

#     acc = accuracy_score(labels, preds)

#     folder_results.append({
#         "CWE": cwe,
#         "Samples": len(group),
#         "Accuracy": acc
#     })

# df_results = pd.DataFrame(folder_results)
# df_results.to_csv("cwe_folder_wise_accuracy.csv", index=False)
# print("\nSaved folder-wise accuracy as cwe_folder_wise_accuracy.csv")

Evaluating CWE: CWE114  (1780 samples)


RuntimeError: MPS backend out of memory (MPS allocated: 16.72 GiB, other allocations: 890.84 MiB, max allowed: 18.13 GiB). Tried to allocate 890.00 MiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

CSV_PATH = "/Users/rohan/Documents/Final year/cwe_project/juliet_cwe_dataset.csv"

df = pd.read_csv(CSV_PATH)
df = df[['code_clean', 'cwe', 'cwe_index']]

# same split used for training
train_df, val_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=42, 
    stratify=df['cwe_index']
)

print("Train samples:", len(train_df))
print("Validation/Test samples:", len(val_df))

# Count per class
train_counts = train_df['cwe'].value_counts()
test_counts = val_df['cwe'].value_counts()

Train samples: 143871
Validation/Test samples: 35968


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = "codet5_juliet_multiclass"   # your model folder

device = "mps" if torch.backends.mps.is_available() else "cpu"
print("Using:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device)

Using: mps


In [ ]:
from torch.utils.data import Dataset, DataLoader

class EvalDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_len
        )
        item = {k: torch.tensor(v) for k, v in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

eval_dataset = EvalDataset(
    val_df['code_clean'], val_df['cwe_index'], tokenizer
)

eval_loader = DataLoader(eval_dataset, batch_size=4)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, precision_recall_fscore_support

def evaluate(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].numpy()

            logits = model(input_ids, attention_mask=attention_mask).logits
            preds = torch.argmax(logits, dim=-1).cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(labels)

    return np.array(all_labels), np.array(all_preds)

y_true, y_pred = evaluate(model, eval_loader, device)

report = classification_report(y_true, y_pred, output_dict=True)

/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/Caskroom/miniconda/base/envs/cwe/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _war

In [ ]:
rows = []

cwe_labels = df.set_index("cwe_index")["cwe"].to_dict()

for idx in sorted(cwe_labels.keys()):
    cwe_name = cwe_labels[idx]
    
    row = {
        "CWE": cwe_name,
        "Train Samples": train_counts.get(cwe_name, 0),
        "Test Samples": test_counts.get(cwe_name, 0),
        "Accuracy": report[str(idx)]["precision"],   # accuracy not provided per-class
        "Precision": report[str(idx)]["precision"],
        "Recall": report[str(idx)]["recall"],
        "F1-score": report[str(idx)]["f1-score"],
    }
    rows.append(row)

excel_df = pd.DataFrame(rows)
excel_df.to_excel("cwe_metrics_summary.xlsx", index=False)

print("✅ Excel file saved as cwe_metrics_summary.xlsx")

✅ Excel file saved as cwe_metrics_summary.xlsx
